<a href="https://colab.research.google.com/github/srinayani123/Arabic_TTS/blob/main/xtts_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y torch torchaudio torchvision TTS transformers numpy -q

In [2]:
!pip install numpy==1.23.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 49.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.7.19 requires torch<2.7,>=1.10, which is not installed.
fastai 2.7.19 requires torchvision>=0.11, which is not installed.
peft 0.15.2 requires torch>=1.13.0, which is not installed.
peft 0.15.2 requires transformers, which is not installed.
accelerate 1.6.0 requires torch>=2.0.0, which is not installed.
jaxlib 0.5.1 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
jax 0.5.2 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
xarray 2025.3.1 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 1.23.5 which is incompatible.
bigframes 2.4.0 requires numpy>=1.24.0, but you have numpy 1.23.5 which is incompa

In [2]:
!pip install torch==2.0.1 torchaudio==2.0.2 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.4/195.4 MB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 57.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, which is not installed.
peft 0.15.2 requires transformers, which is not installed.


In [3]:
!pip install TTS==0.22.0 soundfile datasets transformers==4.36.2

  Using cached TTS-0.22.0-cp311-cp311-manylinux1_x86_64.whl.metadata (21 kB)
  Using cached transformers-4.36.2-py3-none-any.whl.metadata (126 kB)
  Using cached torch-2.7.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached anyascii-0.3.2-py3-none-any.whl.metadata (1.5 kB)
  Using cached pysbd-0.3.4-py3-none-any.whl.metadata (6.1 kB)
  Using cached pandas-1.5.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached trainer-0.0.36-py3-none-any.whl.metadata (8.1 kB)
  Using cached coqpit-0.0.17-py3-none-any.whl.metadata (11 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 52.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6

In [1]:
# STEP 2: LOAD AND RUN XTTS FOR ARABIC
import os
import torch
import time
import numpy as np
import soundfile as sf
from datasets import load_dataset
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.11/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [3]:
import os
import time
import io
#import soundfile as sf
import pandas as pd
import torch
from torch.serialization import safe_globals

#from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts, XttsAudioConfig, XttsArgs
from TTS.config.shared_configs import BaseDatasetConfig
from TTS.utils.manage import ModelManager
from TTS.utils.audio import AudioProcessor # Make sure AudioProcessor is imported

# 🔄 Load XTTS model
print("🔄 Loading XTTS model...")
model_manager = ModelManager()
model_path = model_manager.download_model("tts_models/multilingual/multi-dataset/xtts_v2")[0]

config = XttsConfig()
config.load_json(os.path.join(str(model_path), "config.json"))
model = Xtts.init_from_config(config)

with safe_globals([XttsConfig, XttsAudioConfig, BaseDatasetConfig, XttsArgs]):
    model.load_checkpoint(config, checkpoint_dir=str(model_path))

model.eval()
if torch.cuda.is_available():
    model.cuda()

# 🔊 Load MSA and Najdi samples from SADA parquet
print("🔄 Extracting speaker audio from MSA and Najdi...")
msa_df = pd.read_parquet("/content/train-00000-of-00002.parquet")  # MSA
najdi_df = pd.read_parquet("/content/0017.parquet")  # Najdi

# Extract byte audio and save WAVs
def extract_wav(df_row, output_path):
    audio_bytes = df_row["audio"]["bytes"]
    with io.BytesIO(audio_bytes) as f:
        audio_data, sample_rate = sf.read(f)
    sf.write(output_path, audio_data, sample_rate)
    return output_path

msa_wav = extract_wav(msa_df.loc[0], "msa_speaker.wav")
najdi_wav = extract_wav(najdi_df.loc[0], "najdi_speaker.wav")
print("✅ Saved reference WAVs.")

# 🔁 Get average speaker embedding
# Initialize AudioProcessor directly from the audio config object
print("🔄 Initializing AudioProcessor from config.audio...")

audio_defaults ={"num_mels":80, "fft_size": 2048, "power": 1.5, "preemphasis": 0.0, "signal_norm": True, "symmetric_norm": True, "max_norm": 1.0, "mel_fmin": 0, "mel_fmax": None, "ref_level_db": 20, "do_trim_silence": False, "trim_db": 60, "do_sound_norm": False, "do_amp_to_db_mel": True, "do_amp_to_db_linear": True, "do_rms_norm": False, "db_level": -20 }
for k, v in audio_defaults.items():
    if not hasattr(config.audio, k):
        setattr(config.audio, k, v)
    #setattr(config.audio, k, v)
#ap = AudioProcessor.init_from_config(config)
ap = AudioProcessor(
    sample_rate=config.audio.sample_rate,
    num_mels=config.audio.num_mels,
    fft_size=config.audio.fft_size,
    win_length=None,
    hop_length=None,
    power=config.audio.power,
    preemphasis=config.audio.preemphasis,
    signal_norm=config.audio.signal_norm,
    symmetric_norm=config.audio.symmetric_norm,
    max_norm=config.audio.max_norm,
    mel_fmin=config.audio.mel_fmin,
    mel_fmax=config.audio.mel_fmax,
    ref_level_db=config.audio.ref_level_db,
    do_trim_silence=config.audio.do_trim_silence,
    trim_db=config.audio.trim_db,
    do_sound_norm=config.audio.do_sound_norm,
    do_amp_to_db_mel=config.audio.do_amp_to_db_mel,
    do_amp_to_db_linear=config.audio.do_amp_to_db_linear,
    do_rms_norm=config.audio.do_rms_norm,
    db_level=config.audio.db_level,
    frame_length_ms=48,
    frame_shift_ms=12,
    verbose=False,
)
print("✅ AudioProcessor initialized.")



🔄 Loading XTTS model...
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
🔄 Extracting speaker audio from MSA and Najdi...
✅ Saved reference WAVs.
🔄 Initializing AudioProcessor from config.audio...
✅ AudioProcessor initialized.


In [5]:
import os
import time
import torch
import numpy as np
import soundfile as sf
import pandas as pd

# === Speaker Embedding Utility ===
def get_embedding(wav_path):
    wav, sr = sf.read(wav_path)
    if isinstance(wav, np.ndarray):
        if len(wav.shape) > 1:
            wav = wav.mean(axis=1)
        wav_tensor = torch.tensor(wav, dtype=torch.float32).unsqueeze(0).to(model.device)
        return model.get_speaker_embedding(wav_tensor, sr)
    raise ValueError(f"Unsupported audio format: {type(wav)}")

# === Generate Averaged Speaker Embedding ===
msa_embed = get_embedding(msa_wav)
najdi_embed = get_embedding(najdi_wav)
avg_embed = torch.mean(torch.stack([msa_embed, najdi_embed]), dim=0)
print("✅ Averaged speaker embedding ready.")

# === Arabic Sentences ===
arabic_sentences = {
    "MSA": "مرحبًا، هذه السيارة مزودة بمحرك توربو سعة 2.0 لتر ونظام ملاحة متقدم.",
    "Najdi": "مرحبا، السيارة هاي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن متطور.",
    "Hijazi": "هلا، السيارة دي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن زين.",
    "Gulf": "هلا، السيارة ذي فيها محرك تيربو 2.0 لتر ونظام نافيجيشن ممتاز.",
    "Short": "نعم.",
    "Long": "هذه السيارة مزودة بمحرك توربو سعة 2.0 لتر، ناقل حركة أوتوماتيكي من 8 سرعات، ونظام تثبيت السرعة التكيفي المتقدم بالإضافة إلى حساسات أمامية وخلفية وكاميرا 360 درجة."
}

# === Synthesize Outputs ===
os.makedirs("xtts_outputs", exist_ok=True)
results = []

for dialect, text in arabic_sentences.items():
    print(f"\n🗣️ Generating for: {dialect}")
    start = time.time()

    outputs = model.synthesize(
        text=text,
        config=config, # Pass the full config, synthesize uses it
        speaker_wav= najdi_wav, # Use the averaged embedding
        language="ar"
    )

    latency = time.time() - start
    out_path = f"xtts_outputs/{dialect}.wav"
    sf.write(out_path, outputs["wav"], config.audio.sample_rate)

    duration = len(outputs["wav"]) / config.audio.sample_rate
    print(f"✅ Saved: {out_path} | Latency: {latency:.2f}s | Duration: {duration:.2f}s")

    results.append({
        "Input Type": dialect,
        "Text": text,
        "Latency (s)": round(latency, 2),
        "Audio Length (s)": round(duration, 2),
        "Output Path": out_path
    })

# === Save Metrics ===
df = pd.DataFrame(results)
df.to_csv("xtts_outputs/xtts_eval_metrics.csv", index=False)
print("\n📊 Evaluation metrics saved to xtts_outputs/xtts_eval_metrics.csv")


✅ Averaged speaker embedding ready.

🗣️ Generating for: MSA
✅ Saved: xtts_outputs/MSA.wav | Latency: 46.22s | Duration: 6.87s

🗣️ Generating for: Najdi
✅ Saved: xtts_outputs/Najdi.wav | Latency: 64.16s | Duration: 9.24s

🗣️ Generating for: Hijazi
✅ Saved: xtts_outputs/Hijazi.wav | Latency: 45.78s | Duration: 6.16s

🗣️ Generating for: Gulf
✅ Saved: xtts_outputs/Gulf.wav | Latency: 48.89s | Duration: 6.92s

🗣️ Generating for: Short
✅ Saved: xtts_outputs/Short.wav | Latency: 9.27s | Duration: 0.75s

🗣️ Generating for: Long
✅ Saved: xtts_outputs/Long.wav | Latency: 128.80s | Duration: 17.44s

📊 Evaluation metrics saved to xtts_outputs/xtts_eval_metrics.csv
